# optimizer-state-tensor-buffers — ex2: two-buffer init for an RMSprop-style optimizer

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-state-tensor-buffers`. Running the final beacon cell reports progress against the `Optimizer: Per-param state buffers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Per-param state buffers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-state-tensor-buffers`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-state-tensor-buffers"
DD_SUBTOPIC = "Optimizer: Per-param state buffers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Per-parameter state buffers — quick refresher

Optimizers that need MEMORY across steps (momentum, EMA, second moment) must keep a buffer FOR EACH parameter, allocated at construction time:

```
self.params = list(params)
self.b = [t.zeros_like(p) for p in self.params]    # one buffer per param
```

**Why `zeros_like` not `zeros`.** It mirrors `p`'s `shape`, `dtype`, AND `device` — so a buffer for a `(256, 768)` float16 CUDA weight is itself `(256, 768)` float16 on the same GPU. Initializing as `t.zeros(p.shape)` would silently put the buffer on CPU.

**Why a list, not one big tensor.** Different parameters can have different shapes; you can't flatten them into a single tensor without losing the per-param indexing that `step` relies on. PyTorch's own optimizers use the same per-param list pattern internally.

### Exercise 2 — two-buffer init for an RMSprop-style optimizer

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply per-param `zeros_like` allocation to BOTH the momentum buffer `b` and the squared-gradient EMA buffer `v` in a single optimizer init — matching ARENA's RMSprop spine.
> Keywords: rmsprop-init, two-buffers, v-and-b
> ```

**KCs targeted:** `state-buffer-allocated-via-zeros-like`, `state-buffer-multiple-buffers-per-optimizer`

Implement `Ex2RMSpropInit.__init__(self, params)`. The skeleton of an RMSprop optimizer's init: TWO per-param buffers must be allocated.

1. Materialize `self.params = list(params)` (so the generator case doesn't bite us — see the optimizer-init-list drill).
2. Allocate the momentum buffer: `self.b = [t.zeros_like(p) for p in self.params]`.
3. Allocate the EMA-of-squared-gradients buffer: `self.v = [t.zeros_like(p) for p in self.params]`.
4. (No step / no zero_grad needed — this drill isolates the init step.)

**Critical:** `self.b` and `self.v` must be SEPARATE lists of SEPARATE tensors. A common bug is `self.v = self.b` (alias) — mutating one then mutates the other.

The test passes `model.parameters()` from a small `nn.Linear` stack and verifies that both buffer lists have the right shapes and are independent.

In [ ]:
class Ex2RMSpropInit:
    """Allocate momentum buffer self.b and EMA buffer self.v."""

    def __init__(self, params):
        raise NotImplementedError()


def _test_ex2():
    # Two-layer MLP — diverse param shapes including biases.
    model = t.nn.Sequential(
        t.nn.Linear(8, 16),
        t.nn.ReLU(),
        t.nn.Linear(16, 4),
    )
    expected_shapes = [(16, 8), (16,), (4, 16), (4,)]

    opt = Ex2RMSpropInit(model.parameters())

    assert isinstance(opt.params, list), (
        f'opt.params must be a list (generator must be materialized), got {type(opt.params)}'
    )
    assert len(opt.params) == 4, f'expected 4 params, got {len(opt.params)}'

    # Both buffer lists exist.
    assert hasattr(opt, 'b'), 'missing self.b momentum buffer list'
    assert hasattr(opt, 'v'), 'missing self.v EMA buffer list'
    assert isinstance(opt.b, list) and isinstance(opt.v, list), 'b and v must be lists'
    assert len(opt.b) == 4 and len(opt.v) == 4, (
        f'expected 4-long buffer lists, got len(b)={len(opt.b)}, len(v)={len(opt.v)}'
    )

    # Per-param shape & zero-init.
    for i, (p, b, v, exp_shape) in enumerate(zip(opt.params, opt.b, opt.v, expected_shapes)):
        assert tuple(b.shape) == exp_shape, (
            f'opt.b[{i}].shape {tuple(b.shape)} != {exp_shape}'
        )
        assert tuple(v.shape) == exp_shape, (
            f'opt.v[{i}].shape {tuple(v.shape)} != {exp_shape}'
        )
        assert t.all(b == 0), f'opt.b[{i}] must be all zeros'
        assert t.all(v == 0), f'opt.v[{i}] must be all zeros'
        assert b.requires_grad is False and v.requires_grad is False

    # CRITICAL: b and v must be DIFFERENT tensors, not aliases.
    for i in range(4):
        assert opt.b[i] is not opt.v[i], (
            f'opt.b[{i}] and opt.v[{i}] are the same tensor — did you write self.v = self.b?'
        )
    # Prove non-aliasing by mutation.
    opt.b[0] += 5.0
    assert t.all(opt.v[0] == 0), (
        'after mutating opt.b[0], opt.v[0] should be unchanged; '
        'b and v share storage — did you alias the lists?'
    )
    # Buffers must not share storage WITH the params either.
    for i, (p, b) in enumerate(zip(opt.params, opt.b[1:], )):
        pass  # placeholder — exhaustive check below
    assert opt.b[1].data_ptr() != opt.params[1].data_ptr(), (
        'opt.b[1] shares storage with opt.params[1] — buffers must be independent'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class Ex2RMSpropInit:
    def __init__(self, params):
        self.params = list(params)
        self.b = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]
```

**Why two separate `zeros_like` list-comprehensions, not one.** Each call to `t.zeros_like(p)` allocates a FRESH tensor. If you tried to share allocation — `buf = [t.zeros_like(p) for p in self.params]; self.b = buf; self.v = buf` — both attributes would point to the SAME list of the SAME tensors. Mutating `self.b[0]` in the momentum-update line would silently corrupt `self.v[0]`. Two list-comprehensions guarantees independence.

**The pattern generalizes.** Adam needs `m` and `v` — same two list-comprehensions with names swapped. AdaGrad needs only `G` (sum of squared grads) — one list-comp. The buffer-init block always looks like this; the rest of the optimizer differs.

**Why this is its own drill.** Forgetting the second buffer (or accidentally aliasing it) is a top-3 source of RMSprop/Adam impl bugs in ARENA. Isolating buffer ALLOCATION from buffer UPDATE makes it possible to catch the bug at init time rather than at step time.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()